# 1. Preparation commune du dataset multimodal

## 1.1 Chargement et controle qualite du dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import json
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import tensorflow as tf  # optionnel
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [121]:
csv_path = Path("..") / "train.csv"

try:
    mmc = pd.read_csv(csv_path)
    print("Lecture normale OK")
except pd.errors.ParserError:
    mmc = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")

print("Shape:", mmc.shape)
mmc.head()

Shape: (29918, 3)


,ImageID,Labels,Caption
0,0.jpg,1,Woman in swim suit holding parasol on sunny day.
1,1.jpg,1 19,A couple of men riding horses on top of a gree...
2,2.jpg,1,They are brave for riding in the jungle on tho...
3,3.jpg,8 3 13,a black and silver clock tower at an intersect...
4,4.jpg,8 3 7,A train coming to a stop on the tracks out side.


In [ ]:
# 1) Controle qualite global du dataset
expected_cols = {"ImageID", "Labels", "Caption"}
missing_cols = expected_cols - set(mmc.columns)

print("Colonnes manquantes:", missing_cols if missing_cols else "Aucune")
print("\nValeurs manquantes:")
print(mmc[["ImageID", "Labels", "Caption"]].isna().sum())

print("\nDoublons ImageID:", mmc["ImageID"].duplicated().sum())
print("Shape avant nettoyage:", mmc.shape)

In [ ]:
# 2) Nettoyage minimal des champs bruts
mmc["ImageID"] = mmc["ImageID"].astype(str).str.strip()
mmc["Labels"] = mmc["Labels"].astype(str).str.strip()
mmc["Caption"] = (
    mmc["Caption"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Retirer lignes invalides (vides)
mmc = mmc[
    (mmc["ImageID"] != "") &
    (mmc["Labels"] != "") &
    (mmc["Caption"] != "")
].copy()

print("Shape apres nettoyage:", mmc.shape)
mmc.head(5)

## 1.2 Labels multi-label : distribution et desequilibre

In [ ]:
mmc["Labels_list"] = mmc["Labels"].astype(str).str.split().apply(lambda xs: [int(x) for x in xs])
all_labels = sorted({x for row in mmc["Labels_list"] for x in row})
print("All Labels:", all_labels)

labels_flat = [x for row in mmc["Labels_list"] for x in row]
count_dict = Counter(labels_flat)
print(json.dumps(count_dict, indent=2))

plt.figure(figsize=(12, 6))
plt.hist(labels_flat, bins=len(all_labels), edgecolor="black", alpha=0.7)
plt.xlabel("Labels")
plt.ylabel("Frequency")
plt.title("Distribution des labels")
plt.show()

## 1.3 Verification de la modalite image

In [ ]:

IMAGE_DIR = Path(r"C:\Users\mupps\Desktop\COMP5329S1A2Dataset\data")

num_samples = 4
sampled_df = mmc.sample(num_samples, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(1, num_samples, figsize=(16, 5))

for i, row in sampled_df.iterrows():
    img_path = IMAGE_DIR / row["ImageID"]

    axes[i].axis("off")

    if img_path.exists():
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(
            f'{row["ImageID"]}\nLabels: {row["Labels"]}',
            fontsize=9
        )
        axes[i].text(
            0.5, -0.12, row["Caption"],
            ha="center", va="top",
            fontsize=8,
            wrap=True,
            transform=axes[i].transAxes
        )
    else:
        axes[i].text(
            0.5, 0.5, f"Image introuvable\n{row['ImageID']}",
            ha="center", va="center",
            fontsize=10,
            transform=axes[i].transAxes
        )

plt.tight_layout()
plt.show()

## 1.4 Verification de la modalite texte

In [ ]:
# Important pour la parti LSTM.
mmc["caption_len"] = mmc["Caption"].astype(str).str.split().str.len()

print(f"Longueur moyenne des captions: {mmc['caption_len'].mean():.1f} mots")
print(f"Max: {mmc['caption_len'].max()} | Min: {mmc['caption_len'].min()}")

plt.figure(figsize=(10, 4))
plt.hist(mmc["caption_len"], bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Nombre de mots")
plt.ylabel("Fréquence")
plt.title("Distribution de la longueur des captions")
plt.show()

In [ ]:
lengths = mmc['Caption'].dropna().astype(str).apply(len)
plt.figure()
plt.hist(lengths, bins=25)
plt.xlabel("Caption (Number of Characters)")
plt.ylabel("Frequency")
plt.title("Distribution of Caption ")
plt.show()

## 1.5 Coherence image / texte / labels

In [ ]:
# Cette section est deja en partie couverte par les visualisations ci-dessus.
# On y observe conjointement les images, captions et labels associes.


## 1.6 Split commun train / validation / test

In [ ]:
train_df, temp_df = train_test_split(mmc, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train:      {len(train_df)} exemples")
print(f"Validation: {len(val_df)} exemples")
print(f"Test:       {len(test_df)} exemples")

In [ ]:
# Process Labels (après split)
mlb = MultiLabelBinarizer()

# Important: fit uniquement sur train pour éviter la fuite
train_labels = mlb.fit_transform(train_df["Labels_list"])
val_labels   = mlb.transform(val_df["Labels_list"])
test_labels  = mlb.transform(test_df["Labels_list"])

print("Nb classes:", len(mlb.classes_))
print("Classes:", mlb.classes_)
print("Shape train:", train_labels.shape)
print("Shape val:", val_labels.shape)
print("Shape test:", test_labels.shape)

# Helper tensor (si tu utilises TensorFlow)
def to_float_tensor(y):
    return tf.convert_to_tensor(y, dtype=tf.float32)

y_train = to_float_tensor(train_labels)
y_val   = to_float_tensor(val_labels)
y_test  = to_float_tensor(test_labels)

# 2. Rappel du pipeline image

## 2.1 Preparation de la modalite image

### 2.1.1 Chargement, redimensionnement et normalisation

In [ ]:
# composition de l'image
# A placer dans la partie image preprocessing
def load_image(image_id, target_size=(224, 224)):
    if tf.is_tensor(image_id):
        base_dir = str(IMAGE_DIR).replace("\\", "/")
        img_path = tf.strings.join([base_dir, image_id], separator="/")
        img = tf.io.read_file(img_path)
    else:
        img_path = IMAGE_DIR / str(image_id)
        img = tf.io.read_file(str(img_path))

    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = tf.cast(img, tf.float32) / 255.0
    return img


img = load_image(mmc["ImageID"].iloc[0])

print("Shape (H, W, C):", img.shape)
print("Type:", img.dtype)
print("Min pixel:", tf.reduce_min(img).numpy())
print("Max pixel:", tf.reduce_max(img).numpy())
print("Nb canaux:", img.shape[-1])

plt.imshow(img)
plt.axis("off")
plt.title(f"ImageID: {mmc['ImageID'].iloc[0]}\nLabels: {mmc['Labels'].iloc[0]}", fontsize=9)
plt.show()

arr = img.numpy()
print("Canal R - min/max:", arr[:, :, 0].min(), arr[:, :, 0].max())
print("Canal G - min/max:", arr[:, :, 1].min(), arr[:, :, 1].max())
print("Canal B - min/max:", arr[:, :, 2].min(), arr[:, :, 2].max())


In [ ]:
IMG_SIZE = (224, 224)
IMAGE_BATCH_SIZE = 32

def make_image_dataset(df, labels, batch_size=32, shuffle=False):
    image_ids = df['ImageID'].astype(str).values
    ds = tf.data.Dataset.from_tensor_slices((image_ids, labels.astype(np.float32)))

    def _load(image_id, y):
        image = load_image(image_id, target_size=IMG_SIZE)
        return image, tf.cast(y, tf.float32)

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(len(df), reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_image_ds = make_image_dataset(train_df, train_labels, batch_size=IMAGE_BATCH_SIZE, shuffle=True)
val_image_ds = make_image_dataset(val_df, val_labels, batch_size=IMAGE_BATCH_SIZE, shuffle=False)
test_image_ds = make_image_dataset(test_df, test_labels, batch_size=IMAGE_BATCH_SIZE, shuffle=False)

sample_images, sample_image_labels = next(iter(train_image_ds))
print('Batch images :', sample_images.shape)
print('Batch labels :', sample_image_labels.shape)


## 2.2 Modele image de reference

### 2.2.1 Architecture CNN minimale

In [ ]:
from tensorflow.keras import layers, Model

image_input = layers.Input(shape=(224, 224, 3), name='image')
x = layers.Conv2D(32, 3, padding='same', activation='relu')(image_input)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
image_output = layers.Dense(train_labels.shape[1], activation='sigmoid')(x)

image_model = Model(inputs=image_input, outputs=image_output)
image_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
image_model.summary()


### 2.2.2 Metriques multi-label

In [ ]:
# A completer : F1-score micro/macro, AP par classe, AUC ROC, etc.
# Exemple : sklearn.metrics.f1_score, average_precision_score, roc_auc_score.


### 2.2.3 Diagnostic apprentissage / generalisation

In [ ]:
# Exemple d'entrainement image
# history_image = image_model.fit(
#     train_image_ds,
#     validation_data=val_image_ds,
#     epochs=5
# )

# Puis tracer history_image.history pour visualiser loss / accuracy / val_loss / val_accuracy.


### 2.2.4 Evaluation test du modele image

In [ ]:
# A completer : evaluation sur test_image_ds, matrice de confusion multilabel et interpretation.


## 2.3 Experimentations image a comparer

### 2.3.1 Transfer learning (feature extractor)

In [ ]:
# A completer : utiliser un backbone pre-entraine (ResNet, EfficientNet, DenseNet, etc.)
# en gelant les couches convolutionnelles et en ajoutant une tete de classification.


### 2.3.2 Fine-tuning partiel

In [ ]:
# A completer : degeler certaines couches du backbone et poursuivre l entrainement.


### 2.3.3 Entrainement from scratch

In [ ]:
# A completer : entrainer une architecture CNN sans poids pre-entraines.


## 2.4 Choix du meilleur modele image

In [ ]:
# A completer : comparer les modeles image avec les metriques choisies et justifier le meilleur.


## 2.5 Analyse des predictions image

In [ ]:
# A completer : afficher des images test avec predictions et scores de confiance.
# A completer : ajouter une interpretation type LIME image et analyser les erreurs.


# 3. Rappel du pipeline texte

## 3.1 Preparation de la modalite texte

In [ ]:
captions = train_df['Caption'].dropna().astype(str)
MAX_VOCAB_SIZE = 2000
CAPTION_MAX_LEN = 10
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(captions)

caption_vocab_size = len(tokenizer.word_index)

print("Taille du vocabulaire des captions:", caption_vocab_size)
print("Taille max du vocabulaire retenu pour le modele:", MAX_VOCAB_SIZE)
print("Longueur max des captions pour le modele:", CAPTION_MAX_LEN)

## 3.2 Tokenisation et representation textuelle

In [ ]:
word_counts = tokenizer.word_counts
print(word_counts)
print(f"Nombre de mots uniques: {len(word_counts)}")

frequencies_word = np.array(list(word_counts.values()))
sorted_freqs_word = np.sort(frequencies_word)[::-1]


In [ ]:
total_token_count = np.sum(sorted_freqs_word)
cumulative_counts = np.cumsum(sorted_freqs_word)
results = cumulative_counts / total_token_count

plt.plot(results)
plt.xlabel("Taille du vocabulaire")
plt.ylabel("Couverture cumulative")
plt.title("Couverture cumulative des tokens par frequence")
plt.show()

In [ ]:
# Tokenizer deja entraine plus haut pour eviter une repetition.
effective_vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE)
print("Vocabulaire effectif utilise par le tokenizer:", effective_vocab_size)

In [ ]:
def tokenize_text(text):
    sequences = tokenizer.texts_to_sequences([text])
    return pad_sequences(sequences, maxlen=CAPTION_MAX_LEN, padding='post', truncating='post')

In [ ]:
print(tokenize_text(train_df['Caption'].iloc[15]).shape)
print(tokenize_text(train_df['Caption'].iloc[15]))

## 3.3 Baseline texte machine learning

In [ ]:
# A completer : Logistic Regression, SVM ou Random Forest sur une representation TF-IDF / embeddings.


## 3.4 Modeles texte deep learning : LSTM, BiLSTM, CNN1D

In [ ]:
# A completer : modeles deep learning texte (LSTM, BiLSTM, CNN1D).


## 3.5 Evaluation du modele texte

In [ ]:
# A completer : F1-score, AUC, confusion matrix multilabel, classification report.


## 3.6 Analyse des predictions texte

In [ ]:
# A completer : exemples de captions avec predictions, LimeTextExplainer, analyse des erreurs.


# 4. Fusion multimodale image + texte

## 4.1 Construction des entrees multimodales

In [ ]:
MULTIMODAL_IMAGE_SIZE = (224, 224)
MULTIMODAL_BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def encode_captions(series, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(series.astype(str).tolist())
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_text = encode_captions(train_df["Caption"], tokenizer, CAPTION_MAX_LEN)
X_val_text = encode_captions(val_df["Caption"], tokenizer, CAPTION_MAX_LEN)
X_test_text = encode_captions(test_df["Caption"], tokenizer, CAPTION_MAX_LEN)

print('Shape texte train :', X_train_text.shape)
print('Shape texte val   :', X_val_text.shape)
print('Shape texte test  :', X_test_text.shape)

In [ ]:
def make_multimodal_dataset(df, text_inputs, labels, batch_size=32, shuffle=False):
    image_ids = df['ImageID'].astype(str).values
    captions = df['Caption'].astype(str).values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_ids, captions, text_inputs, labels.astype(np.float32))
    )

    def _load_example(image_id, caption, text_tokens, target):
        image = load_image(image_id, target_size=MULTIMODAL_IMAGE_SIZE)
        features = {
            'image': image,
            'text_tokens': tf.cast(text_tokens, tf.int32),
            'caption': caption,
        }
        return features, tf.cast(target, tf.float32)

    dataset = dataset.map(_load_example, num_parallel_calls=AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset

In [ ]:
train_multimodal_ds = make_multimodal_dataset(
    train_df, X_train_text, train_labels, batch_size=MULTIMODAL_BATCH_SIZE, shuffle=True
)
val_multimodal_ds = make_multimodal_dataset(
    val_df, X_val_text, val_labels, batch_size=MULTIMODAL_BATCH_SIZE, shuffle=False
)
test_multimodal_ds = make_multimodal_dataset(
    test_df, X_test_text, test_labels, batch_size=MULTIMODAL_BATCH_SIZE, shuffle=False
)

sample_features, sample_labels = next(iter(train_multimodal_ds))

print(sample_features["image"].shape)
print(sample_features["text_tokens"].shape)
print(sample_labels.shape)

In [ ]:
print('\nBatch image shape :', sample_features['image'].shape)
print('Batch text shape  :', sample_features['text_tokens'].shape)
print('Batch label shape :', sample_labels.shape)
print('Type image        :', sample_features['image'].dtype)
print('Type text         :', sample_features['text_tokens'].dtype)
print('Type label        :', sample_labels.dtype)

sample_idx = 0
plt.figure(figsize=(4, 4))
plt.imshow(sample_features['image'][sample_idx])
plt.axis('off')
plt.title(sample_features['caption'][sample_idx].numpy().decode('utf-8'))
plt.show()

print('Tokens du premier exemple :', sample_features['text_tokens'][sample_idx].numpy())
print('Caption du premier exemple :', sample_features['caption'][sample_idx].numpy().decode('utf-8'))
print('Labels du premier exemple :', sample_labels[sample_idx].numpy())

## 4.2 Fusion precoce par concatenation de caracteristiques

In [ ]:
# A completer : fusion precoce des representations image et texte dans un meme modele.


## 4.3 Fusion tardive des predictions

In [ ]:
# A completer : fusion tardive des predictions ou logits des meilleurs modeles unimodaux.


## 4.4 Entrainement conjoint image + texte

In [ ]:
# A completer : entrainement conjoint image + texte avec optimisation simultanee.


## 4.5 Comparaison image seule / texte seul / multimodal

In [ ]:
# A completer : comparer image seul, texte seul et multimodal, puis justifier le modele final.


## 4.6 Sauvegarde du modele multimodal
